# Learner Completion Analysis

This notebook documents the analytical and machine-learning component of an academic learning-analytics prototype. It uses **synthetic learner data only** and is intended to demonstrate data preparation, completion-risk modelling, personalised recommendation logic and a sentiment-aware support chatbot.

> **Important:** Model performance shown here is prototype performance on synthetically generated data and should not be interpreted as validated performance on real students.

In [3]:
# STEP 1: Import libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Libraries loaded successfully!")

Libraries loaded successfully!


## Synthetic learner-behaviour dataset

In [4]:
# STEP 2: Create Micro MBA learner behaviour dataset

np.random.seed(42)

n = 612   # Based on the Micro MBA report: 612 registered learners

df = pd.DataFrame({
    "sessions_viewed": np.random.randint(0, 9, n),          # 0 to 8 sessions
    "time_spent_minutes": np.random.randint(0, 481, n),     # up to 8 hours
    "moodlet_logins": np.random.randint(0, 31, n),
    "days_since_last_login": np.random.randint(0, 61, n),
    "engagement_level": np.random.randint(1, 11, n),
    "quiz_attempts": np.random.randint(0, 9, n),
    "rewatched_sessions": np.random.randint(0, 9, n),

    "strategy_score": np.random.randint(0, 101, n),
    "leadership_score": np.random.randint(0, 101, n),
    "sustainability_score": np.random.randint(0, 101, n),
    "finance_score": np.random.randint(0, 101, n),
    "operations_score": np.random.randint(0, 101, n),
    "marketing_score": np.random.randint(0, 101, n),
    "digital_transformation_score": np.random.randint(0, 101, n)
})

df.head()

,sessions_viewed,time_spent_minutes,moodlet_logins,days_since_last_login,engagement_level,quiz_attempts,rewatched_sessions,strategy_score,leadership_score,sustainability_score,finance_score,operations_score,marketing_score,digital_transformation_score
0,6,258,9,19,5,3,2,92,48,75,85,98,51,68
1,3,378,9,31,4,6,5,19,95,0,91,81,67,38
2,7,305,18,35,7,6,2,28,99,54,51,74,25,81
3,4,267,13,49,6,1,4,23,31,21,52,83,74,70
4,6,448,1,3,2,2,6,81,54,39,85,63,1,80


## Dataset preview

In [7]:
# STEP 3: Create completion_status target variable

# Create an overall behaviour score
df["behaviour_score"] = (
    (df["sessions_viewed"] / 8) * 25 +
    (df["time_spent_minutes"] / 480) * 20 +
    (df["moodlet_logins"] / 30) * 15 +
    ((60 - df["days_since_last_login"]) / 60) * 15 +
    (df["engagement_level"] / 10) * 15 +
    (df["quiz_attempts"] / 8) * 10
)

# Learners with stronger behaviour are more likely to complete
df["completion_status"] = 0
df.loc[df["behaviour_score"].rank(pct=True) >= 0.87, "completion_status"] = 1

print(df["completion_status"].value_counts())
df.head()

completion_status
0    532
1     80
Name: count, dtype: int64


,sessions_viewed,time_spent_minutes,moodlet_logins,days_since_last_login,engagement_level,quiz_attempts,rewatched_sessions,strategy_score,leadership_score,sustainability_score,finance_score,operations_score,marketing_score,digital_transformation_score,behaviour_score,completion_status
0,6,258,9,19,5,3,2,92,48,75,85,98,51,68,55.500000,0
1,3,378,9,31,4,6,5,19,95,0,91,81,67,38,50.375000,0
2,7,305,18,35,7,6,2,28,99,54,51,74,25,81,67.833333,1
3,4,267,13,49,6,1,4,23,31,21,52,83,74,70,43.125000,0
4,6,448,1,3,2,2,6,81,54,39,85,63,1,80,57.666667,0


In [8]:
# Define features and target

X = df.drop(
    columns=[
        "completion_status",
        "behaviour_score"
    ]
)

y = df["completion_status"]

print(X.shape)
print(y.shape)

(612, 14)
(612,)


In [9]:
from sklearn.model_selection import train_test_split

## Train/test split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X_train: (489, 14)
X_test: (123, 14)
y_train: (489,)
y_test: (123,)


In [11]:
print(df.shape)
print(X.shape)
print(y.shape)

(612, 16)
(612, 14)
(612,)


In [12]:
# Train Logistic Regression Model

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [13]:
# Make predictions

y_pred = model.predict(X_test)

# Calculate accuracy

from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.967479674796748

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       108
           1       0.87      0.87      0.87        15

    accuracy                           0.97       123
   macro avg       0.92      0.92      0.92       123
weighted avg       0.97      0.97      0.97       123



In [14]:
# STEP 7: Full Personalised Recommendation Engine with Learner Preferences

# Add learner preference to dataset
learning_preferences = ["Video", "Quiz", "Case Study", "Mixed"]

if "learning_preference" not in df.columns:
    df["learning_preference"] = np.random.choice(learning_preferences, size=len(df))

# Module score columns
module_columns = {
    "Strategy": "strategy_score",
    "Leadership and Organisational Culture": "leadership_score",
    "Organisational Sustainability": "sustainability_score",
    "Finance": "finance_score",
    "Operations": "operations_score",
    "Marketing": "marketing_score",
    "Digital Transformation": "digital_transformation_score"
}

# Convert completion probability into learner-friendly likelihood
def get_completion_likelihood(probability):
    if probability >= 0.70:
        return "High"
    elif probability >= 0.40:
        return "Moderate"
    else:
        return "Low"

# Convert completion probability into risk level
def get_risk_level(probability):
    if probability >= 0.70:
        return "Low Risk"
    elif probability >= 0.40:
        return "Medium Risk"
    else:
        return "High Risk"

# Assign priority based on module score
def get_priority(score):
    if score < 50:
        return "High Priority"
    elif score < 70:
        return "Medium Priority"
    elif score < 80:
        return "Low Priority"
    else:
        return "Strong Area"

# Generate actions based on learner preference
def preference_based_actions(module, preference):
    if preference == "Video":
        return [
            f"Rewatch the {module} session recording.",
            f"Pause the video and take notes on key {module} concepts.",
            f"Watch the {module} recording again before retaking the quiz."
        ]

    elif preference == "Quiz":
        return [
            f"Retake the {module} quiz and aim for at least 80%.",
            f"Use quiz feedback to identify which {module} topics need revision.",
            f"Attempt additional practice questions before moving to the next module."
        ]

    elif preference == "Case Study":
        return [
            f"Review the {module} case study or practical examples from the course materials.",
            f"Write a short summary of how the {module} example applies to real business practice.",
            f"Compare the {module} case study with your own experience or workplace context."
        ]

    else:
        return [
            f"Rewatch the {module} session recording.",
            f"Review the {module} examples or case study from the course materials.",
            f"Retake the {module} quiz and aim for at least 80%."
        ]

# Identify strengths, weaknesses, priorities and personalised recommendations
def get_module_recommendations(learner_row):
    module_scores = {
        module: learner_row[column]
        for module, column in module_columns.items()
    }

    sorted_modules = sorted(module_scores.items(), key=lambda x: x[1])
    weakest_modules = sorted_modules[:3]
    strongest_modules = sorted(module_scores.items(), key=lambda x: x[1], reverse=True)[:2]

    priority_dashboard = {
        "High Priority": [],
        "Medium Priority": [],
        "Low Priority": [],
        "Strong Area": []
    }

    for module, score in sorted_modules:
        priority = get_priority(score)
        priority_dashboard[priority].append((module, score))

    preference = learner_row["learning_preference"]

    recommendations = []

    for module, score in weakest_modules:
        priority = get_priority(score)

        if score < 50:
            study_time = "45–60 minutes"
        elif score < 70:
            study_time = "30–45 minutes"
        else:
            study_time = "20–30 minutes"

        if score < 80:
            actions = preference_based_actions(module, preference)
            actions.append(f"Spend {study_time} revising key {module} concepts this week.")

            recommendations.append({
                "module": module,
                "score": score,
                "target": 80,
                "priority": priority,
                "preference": preference,
                "actions": actions
            })

    return strongest_modules, recommendations, priority_dashboard

# Behaviour-based recommendations
def get_engagement_recommendations(learner_row):
    advice = []

    if learner_row["days_since_last_login"] > 14:
        advice.append("Log back into the platform this week because you have been inactive for more than 14 days.")

    if learner_row["sessions_viewed"] < 4:
        advice.append("Complete at least two more recorded sessions this week to improve your learning progress.")

    if learner_row["time_spent_minutes"] < 120:
        advice.append("Increase your learning time. Aim for at least 60 minutes of focused study per week.")

    if learner_row["quiz_attempts"] < 3:
        advice.append("Attempt more quizzes to check your understanding before moving to the next topic.")

    if learner_row["moodlet_logins"] < 5:
        advice.append("Log in 2–3 times per week to maintain consistent engagement.")

    return advice

# Final personalised learning plan
def personalised_learning_plan(learner_row, completion_probability):
    likelihood = get_completion_likelihood(completion_probability)
    risk_level = get_risk_level(completion_probability)

    strongest_modules, module_recommendations, priority_dashboard = get_module_recommendations(learner_row)
    behaviour_recommendations = get_engagement_recommendations(learner_row)

    print("PERSONALISED LEARNING PLAN")
    print("----------------------------------")
    print("Completion Likelihood:", likelihood)
    print("Completion Probability:", round(completion_probability * 100, 2), "%")
    print("Current Risk Level:", risk_level)

    print("\nLearning Analytics Summary:")
    print("Sessions Viewed:", learner_row["sessions_viewed"], "/ 8")
    print("Time Spent:", learner_row["time_spent_minutes"], "minutes")
    print("Moodlet Logins:", learner_row["moodlet_logins"])
    print("Days Since Last Login:", learner_row["days_since_last_login"])
    print("Engagement Level:", learner_row["engagement_level"], "/ 10")
    print("Learning Preference:", learner_row["learning_preference"])

    print("\nProgress Priority Dashboard:")
    for priority, modules in priority_dashboard.items():
        if modules:
            print(f"\n{priority}:")
            for module, score in modules:
                print(f"- {module}: {score}%")

    print("\nStrongest Areas:")
    for module, score in strongest_modules:
        print(f"- {module}: {score}%")

    print("\nRecommended Focus Modules:")
    if module_recommendations:
        for i, rec in enumerate(module_recommendations, 1):
            print(f"\n{i}. {rec['module']} - Current Score: {rec['score']}%")
            print(f"   Priority: {rec['priority']}")
            print(f"   Learning Preference Used: {rec['preference']}")
            print(f"   Target Score: {rec['target']}%")
            for action in rec["actions"]:
                print("   -", action)
    else:
        print("No major module weakness detected. Continue maintaining strong performance.")

    print("\nBehaviour-Based Recommendations:")
    if behaviour_recommendations:
        for advice in behaviour_recommendations:
            print("-", advice)
    else:
        print("Your engagement behaviour looks strong. Keep your current learning routine.")

In [15]:
sample_learner = X_test.iloc[0]

# Add learner preference into X_test sample if needed
sample_index = sample_learner.name
sample_learner = df.loc[sample_index].drop(["completion_status", "behaviour_score"])

completion_probability = model.predict_proba(X_test.iloc[[0]])[0][1]

personalised_learning_plan(sample_learner, completion_probability)

PERSONALISED LEARNING PLAN
----------------------------------
Completion Likelihood: Low
Completion Probability: 0.0 %
Current Risk Level: High Risk

Learning Analytics Summary:
Sessions Viewed: 2 / 8
Time Spent: 387 minutes
Moodlet Logins: 4
Days Since Last Login: 4
Engagement Level: 5 / 10
Learning Preference: Quiz

Progress Priority Dashboard:

High Priority:
- Strategy: 9%

Medium Priority:
- Finance: 53%
- Organisational Sustainability: 60%
- Marketing: 69%

Low Priority:
- Operations: 70%
- Digital Transformation: 72%

Strong Area:
- Leadership and Organisational Culture: 92%

Strongest Areas:
- Leadership and Organisational Culture: 92%
- Digital Transformation: 72%

Recommended Focus Modules:

1. Strategy - Current Score: 9%
   Priority: High Priority
   Learning Preference Used: Quiz
   Target Score: 80%
   - Retake the Strategy quiz and aim for at least 80%.
   - Use quiz feedback to identify which Strategy topics need revision.
   - Attempt additional practice questions befo

In [16]:
sample_learner = X_test.iloc[0]

sample_index = sample_learner.name
sample_learner = df.loc[sample_index]

completion_probability = model.predict_proba(X_test.iloc[[0]])[0][1]

personalised_learning_plan(
    sample_learner,
    completion_probability
)

PERSONALISED LEARNING PLAN
----------------------------------
Completion Likelihood: Low
Completion Probability: 0.0 %
Current Risk Level: High Risk

Learning Analytics Summary:
Sessions Viewed: 2 / 8
Time Spent: 387 minutes
Moodlet Logins: 4
Days Since Last Login: 4
Engagement Level: 5 / 10
Learning Preference: Quiz

Progress Priority Dashboard:

High Priority:
- Strategy: 9%

Medium Priority:
- Finance: 53%
- Organisational Sustainability: 60%
- Marketing: 69%

Low Priority:
- Operations: 70%
- Digital Transformation: 72%

Strong Area:
- Leadership and Organisational Culture: 92%

Strongest Areas:
- Leadership and Organisational Culture: 92%
- Digital Transformation: 72%

Recommended Focus Modules:

1. Strategy - Current Score: 9%
   Priority: High Priority
   Learning Preference Used: Quiz
   Target Score: 80%
   - Retake the Strategy quiz and aim for at least 80%.
   - Use quiz feedback to identify which Strategy topics need revision.
   - Attempt additional practice questions befo

In [17]:
sample_learner = X_test.iloc[0]

sample_index = sample_learner.name
sample_learner = df.loc[sample_index]

completion_probability = model.predict_proba(X_test.iloc[[0]])[0][1]

personalised_learning_plan(
    sample_learner,
    completion_probability
)

PERSONALISED LEARNING PLAN
----------------------------------
Completion Likelihood: Low
Completion Probability: 0.0 %
Current Risk Level: High Risk

Learning Analytics Summary:
Sessions Viewed: 2 / 8
Time Spent: 387 minutes
Moodlet Logins: 4
Days Since Last Login: 4
Engagement Level: 5 / 10
Learning Preference: Quiz

Progress Priority Dashboard:

High Priority:
- Strategy: 9%

Medium Priority:
- Finance: 53%
- Organisational Sustainability: 60%
- Marketing: 69%

Low Priority:
- Operations: 70%
- Digital Transformation: 72%

Strong Area:
- Leadership and Organisational Culture: 92%

Strongest Areas:
- Leadership and Organisational Culture: 92%
- Digital Transformation: 72%

Recommended Focus Modules:

1. Strategy - Current Score: 9%
   Priority: High Priority
   Learning Preference Used: Quiz
   Target Score: 80%
   - Retake the Strategy quiz and aim for at least 80%.
   - Use quiz feedback to identify which Strategy topics need revision.
   - Attempt additional practice questions befo

In [18]:
df.to_csv("../data/micro_mba_dataset_final.csv", index=False)
print("Dataset saved successfully!")

Dataset saved successfully!


In [19]:
# STEP 8: NLP Sentiment-Aware AI Learner Support Chatbot

def detect_sentiment(question):
    question = question.lower()

    negative_words = [
        "stress", "stressed", "worried", "scared", "confused",
        "struggling", "hard", "difficult", "lost", "anxious",
        "fail", "failing", "overwhelmed"
    ]

    positive_words = [
        "happy", "good", "great", "confident", "better",
        "understand", "motivated", "excited"
    ]

    if any(word in question for word in negative_words):
        return "negative"
    elif any(word in question for word in positive_words):
        return "positive"
    else:
        return "neutral"


def learner_chatbot(question, learner_row, completion_probability):
    question_lower = question.lower()

    sentiment = detect_sentiment(question)

    risk = get_risk_level(completion_probability)
    likelihood = get_completion_likelihood(completion_probability)

    weakest_module = min(
        module_columns,
        key=lambda module: learner_row[module_columns[module]]
    )

    weakest_score = learner_row[module_columns[weakest_module]]
    preference = learner_row["learning_preference"]

    if sentiment == "negative":
        emotional_support = (
            "I’m sorry you’re feeling this way 😊. "
            "Let’s take it step by step — you do not need to fix everything at once. "
        )
    elif sentiment == "positive":
        emotional_support = (
            "That’s lovely to hear 😊. Let’s keep building on your progress. "
        )
    else:
        emotional_support = (
            "I’m here to help 😊. "
        )

    if "risk" in question_lower or "why" in question_lower:
        return (
            emotional_support +
            f"You are currently classified as {risk}, with a {likelihood.lower()} completion likelihood. "
            f"Your main focus area is {weakest_module}, where your current score is {weakest_score}%. "
            f"Because your preferred learning style is {preference}, I recommend using "
            f"{preference.lower()}-based support to improve this module first."
        )

    elif "improve" in question_lower or "better" in question_lower:
        return (
            emotional_support +
            f"Your first priority should be {weakest_module}. "
            f"Your current score is {weakest_score}%, and your recommended target is 80%. "
            f"Start by revisiting the {weakest_module} material, then review examples or case studies, "
            "and retake the quiz when you feel ready."
        )

    elif "recommend" in question_lower or "focus" in question_lower or "module" in question_lower:
        return (
            emotional_support +
            f"I recommend focusing on {weakest_module} first because it is currently your lowest-scoring module "
            f"at {weakest_score}%. "
            "A good next step is to review the session recording, revisit the practical example or case study, "
            "and retake the quiz with a target of 80% or above."
        )

    elif "finance" in question_lower:
        return (
            emotional_support +
            f"Your Finance score is {learner_row['finance_score']}%. "
            "For Finance, I recommend reviewing the Finance session recording, looking carefully at the examples "
            "or case study, and retaking the quiz. Try to aim for at least 80% to feel more confident before moving on."
        )

    elif "strategy" in question_lower:
        return (
            emotional_support +
            f"Your Strategy score is {learner_row['strategy_score']}%. "
            "For Strategy, revisit the session recording, review the main strategic concepts, "
            "then test yourself with the quiz again. A good target is 80% or above."
        )

    elif "marketing" in question_lower:
        return (
            emotional_support +
            f"Your Marketing score is {learner_row['marketing_score']}%. "
            "For Marketing, review the session examples, connect the ideas to real business practice, "
            "and retake the quiz if your score is below 80%."
        )

    elif "digital" in question_lower or "transformation" in question_lower:
        return (
            emotional_support +
            f"Your Digital Transformation score is {learner_row['digital_transformation_score']}%. "
            "I recommend revisiting the Digital Transformation session, reviewing examples of digital innovation, "
            "and attempting the quiz again with a target of 80%."
        )

    elif "operations" in question_lower:
        return (
            emotional_support +
            f"Your Operations score is {learner_row['operations_score']}%. "
            "For Operations, spend time reviewing the process examples, then retake the quiz to check your understanding."
        )

    elif "sustainability" in question_lower:
        return (
            emotional_support +
            f"Your Organisational Sustainability score is {learner_row['sustainability_score']}%. "
            "Review the sustainability session and focus on the practical examples before retaking the quiz."
        )

    elif "leadership" in question_lower:
        return (
            emotional_support +
            f"Your Leadership and Organisational Culture score is {learner_row['leadership_score']}%. "
            "Revisit the leadership session, review the organisational culture examples, and aim for 80% in the quiz."
        )

    elif "hello" in question_lower or "hi" in question_lower:
        return (
            "Hello 😊 I’m your AI learning support assistant. "
            "I can help you understand your completion risk, identify weak modules, "
            "and suggest what to study next."
        )

    else:
        return (
            emotional_support +
            "You can ask me things like: "
            "'Why am I high risk?', 'What module should I focus on?', "
            "'How can I improve?', or 'Can you help me with Finance?'"
        )

In [20]:
sample_index = X_test.iloc[0].name
sample_learner = df.loc[sample_index]

completion_probability = model.predict_proba(X_test.iloc[[0]])[0][1]

print(learner_chatbot("I am stressed and confused about Finance", sample_learner, completion_probability))
print()
print(learner_chatbot("Why am I high risk?", sample_learner, completion_probability))
print()
print(learner_chatbot("What module should I focus on?", sample_learner, completion_probability))

I’m sorry you’re feeling this way 😊. Let’s take it step by step — you do not need to fix everything at once. Your Finance score is 53%. For Finance, I recommend reviewing the Finance session recording, looking carefully at the examples or case study, and retaking the quiz. Try to aim for at least 80% to feel more confident before moving on.

I’m here to help 😊. You are currently classified as High Risk, with a low completion likelihood. Your main focus area is Strategy, where your current score is 9%. Because your preferred learning style is Quiz, I recommend using quiz-based support to improve this module first.

I’m here to help 😊. I recommend focusing on Strategy first because it is currently your lowest-scoring module at 9%. A good next step is to review the session recording, revisit the practical example or case study, and retake the quiz with a target of 80% or above.
